In [113]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import csv

In [31]:
import psutil
print(f"RAM dispo: {psutil.virtual_memory().available / 1024**3:.1f} Go")   

RAM dispo: 11.6 Go


In [ ]:
def extraction_data_idf_indus(Fichier1, Fichier2):
    cols_etab = ['siren', 'siret', 'dateCreationEtablissement',
                 'trancheEffectifsEtablissement', 'denominationUsuelleEtablissement', 'dateDernierTraitementEtablissement',
                 'etablissementSiege', 'numeroVoieEtablissement', 'typeVoieEtablissement',
                 'libelleVoieEtablissement', 'codePostalEtablissement',
                 'libelleCommuneEtablissement', 'coordonneeLambertAbscisseEtablissement',
                 'coordonneeLambertOrdonneeEtablissement', 'etatAdministratifEtablissement',
                 'activitePrincipaleEtablissement', 'caractereEmployeurEtablissement', 'codePaysEtrangerEtablissement']

    dep = ('75', '77', '78', '91', '92', '93', '94', '95')
    prefixes_activite = ('10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25',
                          '26', '27', '28', '29', '30', '31', '32', '33', '35', '36', '37', '38', '39', '41', '42', '43',
                            '69', '70', '71', '72', '73', '74', '75')   # SECTION C, D, E, F, M

    pf = pq.ParquetFile(Fichier1)
    n_batches = (pf.metadata.num_rows + 49_999) // 50_000
    batches = []
    for batch in tqdm(pf.iter_batches(batch_size=50_000, columns=cols_etab),
                      total=n_batches, desc="Établissements"):
        df_batch = batch.to_pandas()
        df_batch = df_batch[
        df_batch['codePostalEtablissement'].str.startswith(dep, na=False)
        &
        # df_batch['activitePrincipaleEtablissement'].str.startswith(prefixes_activite, na=False)
        # &
        df_batch['etatAdministratifEtablissement'].str.startswith("A", na=False)
]
        batches.append(df_batch)
    df_etab = pd.concat(batches, ignore_index=True)

    cols_unit = ['siren', 'denominationUniteLegale', 'sexeUniteLegale', 'prenomUsuelUniteLegale',
                 'trancheEffectifsUniteLegale', 'categorieEntreprise', 'categorieJuridiqueUniteLegale']

    pf_2 = pq.ParquetFile(Fichier2)
    n_batches_2 = (pf_2.metadata.num_rows + 49_999) // 50_000
    batches = []
    for batch in tqdm(pf_2.iter_batches(batch_size=50_000, columns=cols_unit),
                      total=n_batches_2, desc="Unités légales"):
        batches.append(batch.to_pandas())
    df_unit = pd.concat(batches, ignore_index=True)

    resultat = pd.merge(df_etab, df_unit, on='siren', how='left')
    return resultat

In [33]:
df_fusion = extraction_data_idf_indus('data/stock-stocketablissement-parquet.parquet', 'data/stock-stockunitelegale-parquet.parquet')

Établissements:   0%|          | 0/882 [00:00<?, ?it/s]

Unités légales: 100%|██████████| 601/601 [00:12<00:00, 48.87it/s]


In [34]:
df_fusion.info()

<class 'pandas.DataFrame'>
RangeIndex: 2966640 entries, 0 to 2966639
Data columns (total 24 columns):
 #   Column                                  Dtype         
---  ------                                  -----         
 0   siren                                   str           
 1   siret                                   str           
 2   dateCreationEtablissement               object        
 3   trancheEffectifsEtablissement           str           
 4   denominationUsuelleEtablissement        str           
 5   dateDernierTraitementEtablissement      datetime64[us]
 6   etablissementSiege                      bool          
 7   numeroVoieEtablissement                 str           
 8   typeVoieEtablissement                   str           
 9   libelleVoieEtablissement                str           
 10  codePostalEtablissement                 str           
 11  libelleCommuneEtablissement             str           
 12  coordonneeLambertAbscisseEtablissement  str          

In [35]:
df_fusion.head()

,siren,siret,dateCreationEtablissement,trancheEffectifsEtablissement,denominationUsuelleEtablissement,dateDernierTraitementEtablissement,etablissementSiege,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,...,etatAdministratifEtablissement,activitePrincipaleEtablissement,caractereEmployeurEtablissement,codePaysEtrangerEtablissement,denominationUniteLegale,sexeUniteLegale,prenomUsuelUniteLegale,trancheEffectifsUniteLegale,categorieEntreprise,categorieJuridiqueUniteLegale
0,005520325,00552032500027,2018-12-20,NN,NaN,2025-12-06 04:31:28,True,36,AVENUE,JUNOT,...,A,70.10Z,N,NaN,ETABLISSEMENTS ADRIEN RIQUIER,NaN,NaN,12,PME,5710
1,005580501,00558050100046,1996-05-01,NN,NaN,2025-12-06 00:46:51,True,93,BOULEVARD,MALESHERBES,...,A,66.30Z,O,NaN,SGN PARIS INVESTISSEMENTS,NaN,NaN,NN,GE,5710
2,005720602,00572060200043,2007-03-19,NN,NaN,2025-12-05 20:27:36,True,14,RUE,MARC BLOCH,...,A,68.20B,N,NaN,SOC IMMOBILIERE MONOPRIX ABBEVILLE,NaN,NaN,NN,PME,5710
3,005720784,00572078400148,2011-01-01,NN,NaN,2025-12-06 04:56:10,False,1,IMPASSE,NICEPHORE NIEPCE,...,A,25.99A,O,NaN,ETABLISSEMENTS DECAYEUX,NaN,NaN,31,PME,5710
4,005720784,00572078400163,2021-12-23,11,ETABLISSEMENTS DECAYEUX,2025-12-05 22:11:10,False,13,RUE,DES INDUSTRIES,...,A,25.99A,O,NaN,ETABLISSEMENTS DECAYEUX,NaN,NaN,31,PME,5710


# Exclusion des entrepreneur individuels

In [69]:
df = df_fusion [df_fusion["categorieJuridiqueUniteLegale"] != 1000]

# Ajout variable nom entreprise

In [ ]:
df["non_entreprise"] = np.where(
    df["denominationUsuelleEtablissement"].isna(),
    df["denominationUniteLegale"],
    df["denominationUsuelleEtablissement"]
)

# Ajout code activité niveau 1

In [ ]:
df["niv1_activitePrincipaleEtablissement"] = df["activitePrincipaleEtablissement"].str[:2] 

# Ajout Lib codes activités

In [140]:
# Charger le fichier de correspondance
naf = pd.read_csv('data/int_courts_naf_rev_2 - NAF rév. 2.csv', sep=',', encoding='utf-8')

# Merge sur le code
df = df.merge(naf[["Code", "Intitule"]],
              left_on="niv1_activitePrincipaleEtablissement",
              right_on="Code",
              how="left")

# Renommer / nettoyer
df = df.rename(columns={"Intitule": "niv1_act_lib"}).drop(columns="Code")

In [143]:
# Charger le fichier de correspondance
naf = pd.read_csv('data/int_courts_naf_rev_2 - NAF rév. 2.csv', sep=',', encoding='utf-8')

# Merge sur le code
df = df.merge(naf[["Code", "Intitule"]],
              left_on="activitePrincipaleEtablissement",
              right_on="Code",
              how="left")

# Renommer / nettoyer
df = df.rename(columns={"Intitule": "act_lib"}).drop(columns="Code")

# Ajout département

In [153]:
df["departement"] = df["codePostalEtablissement"].str[:2] 

In [154]:
df["departement"].value_counts()

departement
75    815069
92    233651
93    182094
78    155005
94    154486
77    139449
95    131343
91    124269
Name: count, dtype: int64

# Supression colonne inutiles

In [146]:
df = df.drop(columns=["denominationUsuelleEtablissement", "denominationUniteLegale",])

In [ ]:
df.to_csv("resultat_idf.csv", index=False, encoding="utf-8")   

# Analyse des valeurs manquantes

In [71]:
df.info()

<class 'pandas.DataFrame'>
Index: 1935366 entries, 0 to 2966639
Data columns (total 24 columns):
 #   Column                                  Dtype         
---  ------                                  -----         
 0   siren                                   str           
 1   siret                                   str           
 2   dateCreationEtablissement               object        
 3   trancheEffectifsEtablissement           str           
 4   denominationUsuelleEtablissement        str           
 5   dateDernierTraitementEtablissement      datetime64[us]
 6   etablissementSiege                      bool          
 7   numeroVoieEtablissement                 str           
 8   typeVoieEtablissement                   str           
 9   libelleVoieEtablissement                str           
 10  codePostalEtablissement                 str           
 11  libelleCommuneEtablissement             str           
 12  coordonneeLambertAbscisseEtablissement  str           
 13

In [149]:
df.isna().sum(axis=0)

siren                                           0
siret                                           0
dateCreationEtablissement                       0
trancheEffectifsEtablissement                   0
dateDernierTraitementEtablissement              0
etablissementSiege                              0
numeroVoieEtablissement                     59238
typeVoieEtablissement                       31420
libelleVoieEtablissement                     1587
codePostalEtablissement                         0
libelleCommuneEtablissement                   472
coordonneeLambertAbscisseEtablissement      83726
coordonneeLambertOrdonneeEtablissement      83726
etatAdministratifEtablissement                  0
activitePrincipaleEtablissement                 0
caractereEmployeurEtablissement              9860
codePaysEtrangerEtablissement             1934894
sexeUniteLegale                           1935366
prenomUsuelUniteLegale                    1935366
trancheEffectifsUniteLegale                     0


In [72]:
df.head()

,siren,siret,dateCreationEtablissement,trancheEffectifsEtablissement,denominationUsuelleEtablissement,dateDernierTraitementEtablissement,etablissementSiege,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,...,etatAdministratifEtablissement,activitePrincipaleEtablissement,caractereEmployeurEtablissement,codePaysEtrangerEtablissement,denominationUniteLegale,sexeUniteLegale,prenomUsuelUniteLegale,trancheEffectifsUniteLegale,categorieEntreprise,categorieJuridiqueUniteLegale
0,005520325,00552032500027,2018-12-20,NN,NaN,2025-12-06 04:31:28,True,36,AVENUE,JUNOT,...,A,70.10Z,N,NaN,ETABLISSEMENTS ADRIEN RIQUIER,NaN,NaN,12,PME,5710
1,005580501,00558050100046,1996-05-01,NN,NaN,2025-12-06 00:46:51,True,93,BOULEVARD,MALESHERBES,...,A,66.30Z,O,NaN,SGN PARIS INVESTISSEMENTS,NaN,NaN,NN,GE,5710
2,005720602,00572060200043,2007-03-19,NN,NaN,2025-12-05 20:27:36,True,14,RUE,MARC BLOCH,...,A,68.20B,N,NaN,SOC IMMOBILIERE MONOPRIX ABBEVILLE,NaN,NaN,NN,PME,5710
3,005720784,00572078400148,2011-01-01,NN,NaN,2025-12-06 04:56:10,False,1,IMPASSE,NICEPHORE NIEPCE,...,A,25.99A,O,NaN,ETABLISSEMENTS DECAYEUX,NaN,NaN,31,PME,5710
4,005720784,00572078400163,2021-12-23,11,ETABLISSEMENTS DECAYEUX,2025-12-05 22:11:10,False,13,RUE,DES INDUSTRIES,...,A,25.99A,O,NaN,ETABLISSEMENTS DECAYEUX,NaN,NaN,31,PME,5710


# Analyse des variable discrètes

In [73]:
df["categorieEntreprise"].value_counts(normalize = True)

categorieEntreprise
PME    0.892896
ETI    0.060910
GE     0.046194
Name: proportion, dtype: float64

In [74]:
df["dateCreationEtablissement"].value_counts(normalize=True)

dateCreationEtablissement
1995-12-25    2.295845e-02
1900-01-01    7.095815e-03
2025-01-01    2.977731e-03
2024-01-01    2.835639e-03
2026-01-01    2.610359e-03
                  ...     
1962-09-10    5.166981e-07
1982-01-24    5.166981e-07
1949-02-11    5.166981e-07
1938-05-19    5.166981e-07
1955-04-04    5.166981e-07
Name: proportion, Length: 22940, dtype: float64

In [75]:
df["activitePrincipaleEtablissement"].value_counts()

activitePrincipaleEtablissement
68.20B    357244
94.99Z    108296
70.22Z     97567
81.10Z     68789
68.20A     66488
           ...  
05.10Z         1
01.64Z         1
01.15Z         1
01.12Z         1
10.51B         1
Name: count, Length: 725, dtype: int64

In [76]:
df["trancheEffectifsEtablissement"].value_counts()

trancheEffectifsEtablissement
NN    1577839
01     147915
02      75689
03      44398
11      37150
12      22997
00      12965
21       8807
22       3992
32       1541
31        819
41        758
42        307
51        167
52         17
53          5
Name: count, dtype: int64

In [77]:
df["caractereEmployeurEtablissement"].value_counts()

caractereEmployeurEtablissement
N    1375286
O     550220
Name: count, dtype: int64

In [78]:
df["categorieJuridiqueUniteLegale"].value_counts()

categorieJuridiqueUniteLegale
5710    669560
5499    383719
6540    329650
9220    222391
9110     69068
         ...  
5658         1
6554         1
6100         1
5670         1
5203         1
Name: count, Length: 219, dtype: int64

In [87]:
df_temp = df[
    (df["denominationUsuelleEtablissement"].isna()) &
    (df["denominationUniteLegale"].isna())
] 

In [93]:
df["non_entreprise"] = np.where(
    df["denominationUsuelleEtablissement"].isna(),
    df["denominationUniteLegale"],
    df["denominationUsuelleEtablissement"]
)

In [98]:
df.info()

<class 'pandas.DataFrame'>
Index: 1935366 entries, 0 to 2966639
Data columns (total 25 columns):
 #   Column                                  Dtype         
---  ------                                  -----         
 0   siren                                   str           
 1   siret                                   str           
 2   dateCreationEtablissement               object        
 3   trancheEffectifsEtablissement           str           
 4   denominationUsuelleEtablissement        str           
 5   dateDernierTraitementEtablissement      datetime64[us]
 6   etablissementSiege                      bool          
 7   numeroVoieEtablissement                 str           
 8   typeVoieEtablissement                   str           
 9   libelleVoieEtablissement                str           
 10  codePostalEtablissement                 str           
 11  libelleCommuneEtablissement             str           
 12  coordonneeLambertAbscisseEtablissement  str           
 13

In [99]:
df_temp = df[
    (df["coordonneeLambertAbscisseEtablissement"].isna())
] 

In [103]:
df_temp["activitePrincipaleEtablissement"].value_counts()

activitePrincipaleEtablissement
68.20B    15838
94.99Z     5835
68.20A     3067
70.22Z     2592
93.12Z     2025
          ...  
01.15Z        1
26.52Z        1
65.20Z        1
17.24Z        1
28.99A        1
Name: count, Length: 639, dtype: int64

In [104]:
print(df_temp["activitePrincipaleEtablissement"].value_counts().to_string())

activitePrincipaleEtablissement
68.20B    15838
94.99Z     5835
68.20A     3067
70.22Z     2592
93.12Z     2025
81.10Z     1864
70.10Z     1737
64.20Z     1316
47.71Z     1315
56.10C     1272
56.10A     1102
90.01Z     1094
62.02A      998
88.99B      987
49.32Z      833
01.11Z      805
66.30Z      770
43.99C      733
71.12B      662
45.11Z      612
41.20A      590
45.20A      590
68.31Z      561
41.10D      560
43.21A      518
68.32A      515
85.59A      474
49.41B      464
62.01Z      450
96.02A      428
47.78C      423
82.99Z      413
43.34Z      412
93.29Z      398
68.10Z      377
43.22A      374
47.91B      373
46.90Z      362
55.10Z      335
47.11B      319
84.11Z      319
35.11Z      317
85.20Z      313
85.59B      312
81.21Z      307
64.19Z      304
88.91A      302
47.78A      289
86.21Z      271
56.29B      269
69.20Z      264
43.32A      263
47.72A      263
64.30Z      262
49.41A      256
96.09Z      252
82.11Z      251
96.02B      248
10.71C      247
81.30Z      244
43.22B  

In [107]:
df["niv1_activitePrincipaleEtablissement"] = df["activitePrincipaleEtablissement"].str[:2] 

In [110]:
print(df["niv1_activitePrincipaleEtablissement"].value_counts().to_string())

niv1_activitePrincipaleEtablissement
68    471258
70    165087
94    121265
47    106703
43     91146
81     84546
56     69263
46     66750
62     58725
64     52305
66     47662
85     44543
49     43491
41     39380
90     38332
93     33076
96     30787
82     28330
71     27177
59     26736
45     26395
88     26017
77     23865
86     20060
69     19842
74     18718
55     16248
58     12811
73     10528
10      9406
35      7792
52      7260
80      7105
84      6485
63      6208
14      5546
01      4824
72      4518
78      4392
87      4263
79      3982
95      3825
32      3371
61      3126
33      2810
42      2180
18      2156
25      2060
53      1885
65      1873
38      1711
02      1478
20      1400
91      1359
26      1121
31      1052
28       896
23       874
37       827
75       708
27       668
60       633
13       604
99       573
36       525
15       512
22       480
16       449
51       428
11       415
29       393
50       384
17       269
39       267
3

In [111]:
df_temp = df[
    (df["coordonneeLambertAbscisseEtablissement"].isna())
] 

In [112]:
print(df_temp["niv1_activitePrincipaleEtablissement"].value_counts().to_string())

niv1_activitePrincipaleEtablissement
68    20477
47     6848
94     6397
70     4536
43     3727
56     3109
93     2855
46     2736
81     2580
85     2061
64     1955
49     1777
41     1556
62     1516
88     1512
90     1466
45     1450
66     1360
01     1286
96     1209
71     1055
82     1038
86     1018
59      692
52      681
55      636
84      636
77      575
74      542
35      536
69      475
10      441
58      327
87      320
73      300
80      265
78      254
72      245
42      237
95      205
33      197
61      179
25      178
38      173
02      166
79      154
63      141
36      112
91      111
14      106
37      101
18       97
23       93
32       87
53       81
65       76
51       71
08       70
26       68
28       66
20       61
31       50
60       39
75       37
22       32
16       31
29       30
27       29
50       26
39       23
24       22
17       18
13       18
30       18
11       16
15       14
99       13
21       12
03        7
92        4
09 